In [ ]:
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from scipy.signal import detrend as signal_detrend
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA


DATA_PATH = Path("data") / "data.xlsx"
OUTPUT_DIR = Path("outputs") / "tables_1_s2"
POSITIVE_SHEET = "Positive"
NEGATIVE_SHEET = "Negative"

CLASSES = ["Lamp oil", "White spirit", "Diesel", "Gasoline"]
MAX_COMPONENTS = 3
CONFIDENCE_LEVEL = 0.9999

TEST_SCENARIOS = {
    "Scenario 1": [
        "T1", "T2", "T4", "T5", "T7", "T10", "T11", "T13", "T14",
        "T6", "T9", "T12", "T15", "Te3", "Te6", "W1", "W2", "W3",
        "L1", "L3", "L7",
    ]
}

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16",
    "SH19", "SH20", "T3", "T6", "T9", "T12", "T15", "Te3", "Te6",
    "Te9", "Te12", "Te15",
}

GAS95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17", "T1", "T4", "T7", "T10",
    "T13", "Te1", "Te4", "Te7", "Te10", "Te13",
}

GAS98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18", "T2", "T5", "T8", "T11",
    "T14", "Te2", "Te5", "Te8", "Te11", "Te14",
}

METHOD_ORDER = [
    "raw",
    "baseline",
    "detrend",
    "normalisation",
    "snv",
    "msc",
    "sg1_p2_w3",
    "sg2_p2_w3",
    "snv+sg1",
    "msc+sg1",
]

PRETTY_NAMES = {
    "raw": "Raw",
    "baseline": "Baseline correction",
    "detrend": "Detrending",
    "normalisation": "Normalisation",
    "snv": "SNV",
    "msc": "MSC",
    "sg1_p2_w3": "SG1 (p2, w3)",
    "sg2_p2_w3": "SG2 (p2, w3)",
    "snv+sg1": "SNV + SG1",
    "msc+sg1": "MSC + SG1",
}


def build_metadata(spectra: pd.DataFrame) -> pd.DataFrame:
    roots = spectra.index.to_series().astype(str).str.split("-", n=1).str[0]
    classes = []

    for root in roots:
        if root.startswith("L"):
            classes.append("Lamp oil")
        elif root.startswith("W"):
            classes.append("White spirit")
        elif root in DIESEL_ROOTS:
            classes.append("Diesel")
        elif root in GAS95_ROOTS or root in GAS98_ROOTS:
            classes.append("Gasoline")
        elif root.startswith("B"):
            classes.append("Brandspiritus")
        else:
            raise ValueError(f"Unknown root code: {root}")

    return pd.DataFrame(
        {"root": roots.to_numpy(), "simca_class": classes},
        index=spectra.index,
    )


def is_brandspiritus_root(roots: pd.Series) -> pd.Series:
    return roots.astype(str).str.startswith("B")


def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, np.ndarray]:
    positive = pd.read_excel(DATA_PATH, sheet_name=POSITIVE_SHEET)
    negative = pd.read_excel(DATA_PATH, sheet_name=NEGATIVE_SHEET)
    positive = positive.set_index(positive.columns[0])
    negative = negative.set_index(negative.columns[0])
    positive.index = positive.index.astype(str)
    negative.index = negative.index.astype(str)
    positive.columns = positive.columns.astype(str)
    negative.columns = negative.columns.astype(str)

    spectral_columns = sorted(positive.columns, key=lambda column: float(column))
    positive = positive.loc[:, spectral_columns]
    negative = negative.loc[:, spectral_columns]
    metadata = build_metadata(positive)

    positive_excluded = is_brandspiritus_root(metadata["root"])
    positive = positive.loc[~positive_excluded].copy()
    metadata = metadata.loc[~positive_excluded].copy()

    negative_roots = negative.index.to_series().str.split("-", n=1).str[0]
    negative = negative.loc[~is_brandspiritus_root(negative_roots)].copy()

    if is_brandspiritus_root(metadata["root"]).any():
        raise RuntimeError("Brandspiritus samples remain in the positive analysis data.")
    remaining_negative_roots = negative.index.to_series().str.split("-", n=1).str[0]
    if is_brandspiritus_root(remaining_negative_roots).any():
        raise RuntimeError("Brandspiritus samples remain in the negative analysis data.")

    wavelengths = np.asarray([float(column) for column in spectral_columns], dtype=float)
    return positive, negative, metadata, wavelengths


def preproc_raw(df: pd.DataFrame) -> pd.DataFrame:
    return df.copy()


def preproc_normalisation(df: pd.DataFrame) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return pd.DataFrame(values / norms, index=df.index, columns=df.columns)


def preproc_snv(df: pd.DataFrame) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    means = values.mean(axis=1, keepdims=True)
    standard_deviations = values.std(axis=1, ddof=1, keepdims=True)
    standard_deviations[standard_deviations == 0.0] = 1.0
    corrected = (values - means) / standard_deviations
    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_msc(df: pd.DataFrame, reference: np.ndarray) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    reference = np.asarray(reference, dtype=float)
    design = np.column_stack([reference, np.ones_like(reference)])
    corrected = np.empty_like(values)

    for index, row in enumerate(values):
        slope, intercept = np.linalg.lstsq(design, row, rcond=None)[0]
        corrected[index] = (row - intercept) / slope

    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_baseline_endpoints(
    df: pd.DataFrame,
    wavelengths: np.ndarray,
) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    denominator = wavelengths[-1] - wavelengths[0]
    if denominator == 0:
        denominator = 1.0
    corrected = np.empty_like(values)

    for index, row in enumerate(values):
        slope = (row[-1] - row[0]) / denominator
        baseline = row[0] + slope * (wavelengths - wavelengths[0])
        corrected[index] = row - baseline

    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_detrend(df: pd.DataFrame) -> pd.DataFrame:
    corrected = signal_detrend(df.to_numpy(dtype=float), axis=1, type="linear")
    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_sg(
    df: pd.DataFrame,
    polyorder: int,
    window_length: int,
    deriv: int,
) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    if window_length % 2 == 0:
        window_length += 1
    if window_length > values.shape[1]:
        window_length = values.shape[1] if values.shape[1] % 2 == 1 else values.shape[1] - 1
    filtered = savgol_filter(
        values,
        window_length=window_length,
        polyorder=polyorder,
        deriv=deriv,
        axis=1,
        mode="interp",
    )
    return pd.DataFrame(filtered, index=df.index, columns=df.columns)


def get_preprocessing_pipelines(
    wavelengths: np.ndarray,
    msc_reference: np.ndarray,
) -> dict[str, Callable[[pd.DataFrame], pd.DataFrame]]:
    def sg1(df: pd.DataFrame) -> pd.DataFrame:
        return preproc_sg(df, polyorder=2, window_length=3, deriv=1)

    def sg2(df: pd.DataFrame) -> pd.DataFrame:
        return preproc_sg(df, polyorder=2, window_length=3, deriv=2)

    return {
        "raw": preproc_raw,
        "baseline": lambda df: preproc_baseline_endpoints(df, wavelengths),
        "detrend": preproc_detrend,
        "normalisation": preproc_normalisation,
        "snv": preproc_snv,
        "msc": lambda df: preproc_msc(df, msc_reference),
        "sg1_p2_w3": sg1,
        "sg2_p2_w3": sg2,
        "snv+sg1": lambda df: sg1(preproc_snv(df)),
        "msc+sg1": lambda df: sg1(preproc_msc(df, msc_reference)),
    }


class SimcaClassModel:
    def __init__(
        self,
        pca: PCA,
        mean: np.ndarray,
        eigenvalues: np.ndarray,
        training_t2: np.ndarray,
        training_q: np.ndarray,
    ) -> None:
        self.pca = pca
        self.mean = mean
        self.eigenvalues = eigenvalues
        self.training_t2 = training_t2
        self.training_q = training_q


def fit_simca_class(values: np.ndarray) -> SimcaClassModel:
    n_components = min(MAX_COMPONENTS, values.shape[0], values.shape[1])
    mean = values.mean(axis=0)
    centred = values - mean
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(centred)
    reconstructed = pca.inverse_transform(scores)
    residuals = centred - reconstructed
    eigenvalues = pca.explained_variance_
    training_t2 = np.sum((scores ** 2) / eigenvalues, axis=1)
    training_q = np.sum(residuals ** 2, axis=1)
    return SimcaClassModel(
        pca,
        mean,
        eigenvalues,
        training_t2,
        training_q,
    )


def fit_all_class_models(
    positive: pd.DataFrame,
    metadata: pd.DataFrame,
) -> dict[str, SimcaClassModel]:
    models = {}
    for class_name in CLASSES:
        class_values = positive.loc[
            metadata["simca_class"] == class_name
        ].to_numpy(dtype=float)
        if class_values.shape[0] == 0:
            raise ValueError(f"No development samples available for class: {class_name}")
        models[class_name] = fit_simca_class(class_values)
    return models


def score_simca_class(
    values: np.ndarray,
    model: SimcaClassModel,
) -> tuple[np.ndarray, np.ndarray]:
    centred = values - model.mean
    scores = model.pca.transform(centred)
    reconstructed = model.pca.inverse_transform(scores)
    residuals = centred - reconstructed
    t2 = np.sum((scores ** 2) / model.eigenvalues, axis=1)
    q = np.sum(residuals ** 2, axis=1)
    return t2, q


def compute_limits(
    models: dict[str, SimcaClassModel],
) -> dict[str, tuple[float, float]]:
    alpha = 1.0 - CONFIDENCE_LEVEL
    return {
        class_name: (
            np.quantile(model.training_t2, 1.0 - alpha),
            np.quantile(model.training_q, 1.0 - alpha),
        )
        for class_name, model in models.items()
    }


def classify_with_simca(
    spectra: pd.DataFrame,
    models: dict[str, SimcaClassModel],
    limits: dict[str, tuple[float, float]],
) -> pd.DataFrame:
    values = spectra.to_numpy(dtype=float)
    membership = pd.DataFrame(False, index=spectra.index, columns=CLASSES)

    for class_name in CLASSES:
        t2, q = score_simca_class(values, models[class_name])
        t2_limit, q_limit = limits[class_name]
        membership[class_name] = (t2 <= t2_limit) & (q <= q_limit)

    return membership


def compute_reduced_distances(
    spectra: pd.DataFrame,
    models: dict[str, SimcaClassModel],
    limits: dict[str, tuple[float, float]],
) -> pd.DataFrame:
    values = spectra.to_numpy(dtype=float)
    distances = pd.DataFrame(index=spectra.index, columns=CLASSES, dtype=float)

    for class_name in CLASSES:
        t2, q = score_simca_class(values, models[class_name])
        t2_limit, q_limit = limits[class_name]
        distances[class_name] = np.sqrt(
            (t2 / t2_limit) ** 2 + (q / q_limit) ** 2
        )

    return distances


def assert_three_components(models: dict[str, SimcaClassModel]) -> None:
    incorrect = [
        f"{class_name}: {int(models[class_name].pca.n_components_)} PCs"
        for class_name in CLASSES
        if int(models[class_name].pca.n_components_) != MAX_COMPONENTS
    ]
    if incorrect:
        raise RuntimeError("Incorrect PC counts: " + "; ".join(incorrect))


def collect_explained_variance(
    scenario_name: str,
    method: str,
    models: dict[str, SimcaClassModel],
    development_metadata: pd.DataFrame,
) -> list[dict]:
    records = []

    for class_name in CLASSES:
        model = models[class_name]
        explained = model.pca.explained_variance_ratio_
        cumulative = np.cumsum(explained)
        number_of_samples = int(
            (development_metadata["simca_class"] == class_name).sum()
        )
        for pc_index, (ratio, accumulated) in enumerate(
            zip(explained, cumulative),
            start=1,
        ):
            records.append(
                {
                    "Scenario": scenario_name,
                    "Preprocessing method": PRETTY_NAMES[method],
                    "Class model": class_name,
                    "PC": f"PC{pc_index}",
                    "n_development_samples_class": number_of_samples,
                    "n_components_used": int(model.pca.n_components_),
                    "explained_variance_ratio": float(ratio),
                    "accumulated_explained_variance": float(accumulated),
                }
            )

    return records


def confidence_label() -> str:
    return "99.99%"


def split_positive_by_test_roots(
    all_positive: pd.DataFrame,
    all_metadata: pd.DataFrame,
    test_roots: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    test_root_set = set(test_roots)
    available_roots = set(all_metadata["root"].unique())
    missing_roots = sorted(test_root_set - available_roots)
    if missing_roots:
        raise ValueError(f"External-test roots missing from Positive sheet: {missing_roots}")
    if is_brandspiritus_root(all_metadata["root"]).any():
        raise RuntimeError("Brandspiritus samples remain before the scenario split.")

    test_mask = all_metadata["root"].isin(test_root_set)
    development = all_positive.loc[~test_mask].copy()
    development_metadata = all_metadata.loc[~test_mask].copy()
    external = all_positive.loc[test_mask].copy()
    external_metadata = all_metadata.loc[test_mask].copy()

    if development.empty:
        raise ValueError("The development set is empty.")
    if external.empty:
        raise ValueError("The external-test set is empty.")

    return development, development_metadata, external, external_metadata


def evaluate_scenario(
    scenario_name: str,
    test_roots: list[str],
    all_positive: pd.DataFrame,
    negative: pd.DataFrame,
    all_metadata: pd.DataFrame,
    wavelengths: np.ndarray,
) -> tuple[dict[str, list[dict]], list[dict], list[dict]]:
    development, development_metadata, external, external_metadata = (
        split_positive_by_test_roots(all_positive, all_metadata, test_roots)
    )
    msc_reference = development.to_numpy(dtype=float).mean(axis=0)
    pipelines = get_preprocessing_pipelines(wavelengths, msc_reference)
    class_results = {class_name: [] for class_name in CLASSES}
    explained_variance_records = []
    error_records = []

    for method in METHOD_ORDER:
        preprocessing = pipelines[method]
        prepared_development = preprocessing(development)
        prepared_external = preprocessing(external)
        prepared_negative = preprocessing(negative)
        models = fit_all_class_models(prepared_development, development_metadata)
        assert_three_components(models)
        explained_variance_records.extend(
            collect_explained_variance(
                scenario_name,
                method,
                models,
                development_metadata,
            )
        )

        limits = compute_limits(models)
        negative_distances = compute_reduced_distances(
            prepared_negative,
            models,
            limits,
        )
        external_distances = compute_reduced_distances(
            prepared_external,
            models,
            limits,
        )
        external_membership = classify_with_simca(
            prepared_external,
            models,
            limits,
        )
        negative_membership = classify_with_simca(
            prepared_negative,
            models,
            limits,
        )
        negative_values = prepared_negative.to_numpy(dtype=float)
        external_values = prepared_external.to_numpy(dtype=float)

        for class_name in CLASSES:
            model = models[class_name]
            t2_limit, q_limit = limits[class_name]
            negative_t2, negative_q = score_simca_class(negative_values, model)
            external_t2, external_q = score_simca_class(external_values, model)
            negative_t2 = pd.Series(negative_t2, index=prepared_negative.index)
            negative_q = pd.Series(negative_q, index=prepared_negative.index)
            external_t2 = pd.Series(external_t2, index=prepared_external.index)
            external_q = pd.Series(external_q, index=prepared_external.index)

            accepted_negative = negative_membership[class_name].to_numpy(dtype=bool)
            false_positive_rate = float(accepted_negative.mean())
            class_mask = external_metadata["simca_class"] == class_name
            number_of_external_samples = int(class_mask.sum())
            if number_of_external_samples:
                accepted_external = external_membership.loc[
                    class_mask,
                    class_name,
                ].to_numpy(dtype=bool)
                false_negative_rate = float(1.0 - accepted_external.mean())
            else:
                false_negative_rate = np.nan

            class_distances = negative_distances[class_name].to_numpy(dtype=float)
            class_results[class_name].append(
                {
                    "Scenario": scenario_name,
                    "Confidence level": confidence_label(),
                    "Preprocessing method": PRETTY_NAMES[method],
                    "Class model": class_name,
                    "n_external_positive_class": number_of_external_samples,
                    "n_negative": len(prepared_negative),
                    "FPR": false_positive_rate,
                    "FNR": false_negative_rate,
                    "dij_mean": float(class_distances.mean()),
                    "dij_min": float(class_distances.min()),
                }
            )

            false_positive_ids = negative_membership.index[
                negative_membership[class_name].to_numpy(dtype=bool)
            ]
            for sample_id in false_positive_ids:
                error_records.append(
                    {
                        "Scenario": scenario_name,
                        "Confidence level": confidence_label(),
                        "Preprocessing method": PRETTY_NAMES[method],
                        "Error type": "FPR",
                        "Sample ID": sample_id,
                        "True class": "Negative",
                        "Class model": class_name,
                        "Predicted/accepted as": class_name,
                        "T2": float(negative_t2.loc[sample_id]),
                        "Q": float(negative_q.loc[sample_id]),
                        "T2crit": float(t2_limit),
                        "Qcrit": float(q_limit),
                        "dij": float(negative_distances.loc[sample_id, class_name]),
                    }
                )

            false_negative_mask = (
                class_mask
                & ~external_membership[class_name].to_numpy(dtype=bool)
            )
            false_negative_ids = external_metadata.index[false_negative_mask]
            for sample_id in false_negative_ids:
                error_records.append(
                    {
                        "Scenario": scenario_name,
                        "Confidence level": confidence_label(),
                        "Preprocessing method": PRETTY_NAMES[method],
                        "Error type": "FNR",
                        "Sample ID": sample_id,
                        "True class": class_name,
                        "Class model": class_name,
                        "Predicted/accepted as": "Rejected by true class model",
                        "T2": float(external_t2.loc[sample_id]),
                        "Q": float(external_q.loc[sample_id]),
                        "T2crit": float(t2_limit),
                        "Qcrit": float(q_limit),
                        "dij": float(external_distances.loc[sample_id, class_name]),
                    }
                )

    return class_results, explained_variance_records, error_records


def write_class_results(
    all_results: dict[str, list[dict]],
    output_path: Path,
) -> None:
    method_rank = {
        PRETTY_NAMES[method]: index
        for index, method in enumerate(METHOD_ORDER)
    }
    scenario_rank = {
        scenario: index
        for index, scenario in enumerate(TEST_SCENARIOS)
    }

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for class_name in CLASSES:
            table = pd.DataFrame(all_results[class_name])
            table["scenario_rank"] = table["Scenario"].map(scenario_rank)
            table["method_rank"] = table["Preprocessing method"].map(method_rank)
            table = (
                table.sort_values(["scenario_rank", "method_rank"])
                .drop(columns=["scenario_rank", "method_rank"])
                .reset_index(drop=True)
            )
            table.to_excel(writer, sheet_name=class_name, index=False)


def write_explained_variance(
    records: list[dict],
    output_path: Path,
) -> None:
    table = pd.DataFrame(records)
    scenario_rank = {
        scenario: index
        for index, scenario in enumerate(TEST_SCENARIOS)
    }
    method_rank = {
        PRETTY_NAMES[method]: index
        for index, method in enumerate(METHOD_ORDER)
    }
    class_rank = {class_name: index for index, class_name in enumerate(CLASSES)}
    table["scenario_rank"] = table["Scenario"].map(scenario_rank)
    table["method_rank"] = table["Preprocessing method"].map(method_rank)
    table["class_rank"] = table["Class model"].map(class_rank)
    table["pc_rank"] = table["PC"].str.replace("PC", "", regex=False).astype(int)
    table = (
        table.sort_values(["scenario_rank", "method_rank", "class_rank", "pc_rank"])
        .drop(columns=["scenario_rank", "method_rank", "class_rank", "pc_rank"])
        .reset_index(drop=True)
    )

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        table.to_excel(writer, sheet_name="All", index=False)
        for class_name in CLASSES:
            class_table = table.loc[table["Class model"] == class_name].reset_index(drop=True)
            class_table.to_excel(writer, sheet_name=class_name, index=False)


def write_error_samples(records: list[dict], output_path: Path) -> None:
    columns = [
        "Scenario",
        "Confidence level",
        "Preprocessing method",
        "Error type",
        "Sample ID",
        "True class",
        "Class model",
        "Predicted/accepted as",
        "T2",
        "Q",
        "T2crit",
        "Qcrit",
        "dij",
    ]
    table = pd.DataFrame(records, columns=columns)

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        table.to_excel(writer, sheet_name="All_errors", index=False)
        table.loc[table["Error type"] == "FPR"].to_excel(
            writer,
            sheet_name="FPR",
            index=False,
        )
        table.loc[table["Error type"] == "FNR"].to_excel(
            writer,
            sheet_name="FNR",
            index=False,
        )


def run() -> None:
    positive, negative, metadata, wavelengths = load_data()
    all_class_results = {class_name: [] for class_name in CLASSES}
    all_explained_variance = []
    all_error_samples = []

    for scenario_name, test_roots in TEST_SCENARIOS.items():
        class_results, explained_variance, error_samples = evaluate_scenario(
            scenario_name,
            test_roots,
            positive,
            negative,
            metadata,
            wavelengths,
        )
        for class_name, records in class_results.items():
            all_class_results[class_name].extend(records)
        all_explained_variance.extend(explained_variance)
        all_error_samples.extend(error_samples)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    write_explained_variance(
        all_explained_variance,
        OUTPUT_DIR / "SIMCA_accumulated_explained_variance.xlsx",
    )
    write_class_results(
        all_class_results,
        OUTPUT_DIR / "SIMCA_result_FPR_FNR_dij.xlsx",
    )
    write_error_samples(
        all_error_samples,
        OUTPUT_DIR / "SIMCA_FPR_FNR_sample.xlsx",
    )


if __name__ == "__main__":
    run()
